In [ ]:
#@title Cell 55.1 — Objective and validation safeguards
from IPython.display import display, Markdown

display(Markdown(r"""
# NB55 — Training-only recalibration of BMC-N breakpoint probabilities (v0.1)

**Question:** Does a small probability recalibration improve agreement between predicted and observed threshold-exceedance frequencies in **independently held-out** pathogens?

**Unchanged:** Model C point predictions; BMC-N predictive MIC distributions and the NB54 uncalibrated probabilities; antibiotic-specific EUCAST threshold scenarios from NB53; all previously approved manuscript and supplementary outputs.

### Pre-specified recalibration

For held-out predicted threshold-exceedance probability $p$, use one antibiotic- and outer-fold-specific correction:

$$p_{\mathrm{recal}}=\operatorname{logit}^{-1}\left[\operatorname{logit}(p)+\alpha_{a,k}\right].$$

Only **one intercept** $\alpha_{a,k}$ is estimated; the slope is fixed at 1. This preserves the ranking of pathogens. It is not a recalibration of the full MIC predictive distribution—only of the probability of a **specified breakpoint-exceedance event**.

**Training for each outer fold:** Start from NB40's training-only three-fold, pathogen-out Model C predictions. For each inner fold, fit the *original BMC-N error distribution* on residuals from the **other two inner folds**, then calculate threshold-exceedance probabilities for the excluded inner fold. Fit $\alpha_{a,k}$ to these training-only probability–outcome pairs, using a fixed $\mathcal N(0,2^2)$ penalty to stabilise sparse threshold events.

**Independent evaluation:** Apply the fitted intercept to **unchanged NB54 held-out probabilities**. Outer-test MICs are used **only for evaluation**, never for selecting or estimating $\alpha$.

Important qualification: NB40's inner Model C cross-fits were not constructed as a completely nested calibration trial. Although we exclude each inner group directly from its BMC posterior fit, some other inner residuals were generated using Model C fits trained on that group's pathogens. Consequently, these training-side pairs are not strictly independent; **the external-to-training outer-test evaluation remains the unbiased performance check**.

Report raw **and** recalibrated: Brier score, log loss, calibration-in-the-large, reliability bins and the observed frequencies in $p\ge0.90$ and $p\le0.10$ groups. Recalibration is useful only if it actually improves held-out results.

**Clinical qualification:** Thresholds may be indication-restricted, while infection indication is absent. Results do not establish clinical resistance classification for every patient or replace laboratory AST.
"""))


In [ ]:
#@title Cell 55.2 — Mount Drive and locate exact earlier notebook outputs
import os, json, hashlib
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.special import expit, logit, logsumexp, ndtr
from scipy.optimize import brentq
from sklearn.metrics import roc_auc_score
import matplotlib.pyplot as plt

try:
    from google.colab import drive
    if not Path('/content/drive/MyDrive').is_dir():
        drive.mount('/content/drive')
except ImportError:
    pass

ROOT = Path(os.environ.get('BMC_PROJECT_ROOT','/content/drive/MyDrive/RefMIC'))
if not ROOT.is_dir():
    raise FileNotFoundError(f'RefMIC folder not found: {ROOT}. For local Jupyter, set BMC_PROJECT_ROOT.')
RESULTS = ROOT / '03_Results'
OUT = RESULTS / '55_BMC_Probability_Training_Only_Recalibration'
OUT.mkdir(parents=True,exist_ok=True)

FILES = {
    'nb40_train': ('40_BMC_Training_Only_Calibration', '40_BMC_training_only_inner_predictions_all_outer_folds.csv.gz'),
    'nb54_test': ('54_BMC_Breakpoint_Exceedance_Probability', '54_BMCN_Breakpoint_Probabilities_HeldOut.csv.gz'),
    'nb53_cutoffs': ('53_BMC_Resistance_Threshold_Coverage', '53_Supplementary_Resistance_Coverage_18_Antibiotics.csv'),
}

def checksum(path):
    sha=hashlib.sha256()
    with open(path,'rb') as f:
        for block in iter(lambda:f.read(4*1024*1024),b''):
            sha.update(block)
    return sha.hexdigest()

def locate_output(folder, filename):
    preferred=RESULTS/folder/filename
    if preferred.is_file():
        return preferred
    matches=sorted(p for p in ROOT.rglob(filename) if p.is_file())
    if not matches:
        raise FileNotFoundError(f'Missing {filename}. Expected {preferred}; run the relevant notebook save cell.')
    if len(matches)>1 and len({checksum(p) for p in matches})!=1:
        raise ValueError(f'Multiple different {filename} files found; resolve before continuing:\n'+ '\n'.join(map(str,matches)))
    return matches[0]

PATHS={key:locate_output(*value) for key,value in FILES.items()}
for k,v in PATHS.items():
    print(k,':',v)
print('Output:',OUT)


In [ ]:
#@title Cell 55.3 — Load training data and independently held-out NB54 probabilities
calib=pd.read_csv(PATHS['nb40_train'],compression='gzip')
test=pd.read_csv(PATHS['nb54_test'],compression='gzip')
cutoffs=pd.read_csv(PATHS['nb53_cutoffs'])

req_cal={'outer_fold_for_calibration','inner_fold','interaction_row','biosample','antibiotic',
         'observed_log2_mic','inner_oof_predicted_log2_mic','training_residual_log2'}
req_test={'outer_fold','interaction_row','biosample','antibiotic','observed_log2_mic',
          'model_c_outer_test_predicted_log2_mic','prob_mic_exceeds_r','observed_exceeds_r',
          'r_cutoff_mg_L_exclusive'}
req_cut={'antibiotic','r_cutoff_mg_L_exclusive','n_tested'}
for name,table,required in [('NB40',calib,req_cal),('NB54',test,req_test),('NB53',cutoffs,req_cut)]:
    missing=required-set(table.columns)
    if missing:
        raise ValueError(f'{name} missing required columns: {sorted(missing)}')
for table in (calib,test,cutoffs):
    table['antibiotic']=table.antibiotic.astype(str).str.strip().str.lower()

assert len(calib)==201840, f'Unexpected NB40 training size: {len(calib)}'
assert not calib.duplicated(['outer_fold_for_calibration','interaction_row']).any()
assert not test.interaction_row.duplicated().any()
assert not test.duplicated(['biosample','antibiotic']).any()
assert len(cutoffs)==18 and cutoffs.antibiotic.is_unique
assert set(test.antibiotic)==set(cutoffs.antibiotic)
assert len(test)==int(cutoffs.n_tested.sum())
assert set(test.outer_fold)==set(range(1,6))
assert set(calib.inner_fold)=={1,2,3}
assert (calib.groupby(['outer_fold_for_calibration','biosample']).inner_fold.nunique()==1).all()

# Confirm exact observed outcome; it is strictly reserved for evaluating test predictions.
true_event=(test.observed_log2_mic.to_numpy(float) > np.log2(test.r_cutoff_mg_L_exclusive.to_numpy(float)))
stored_event=test.observed_exceeds_r.astype(str).str.lower().map({'true':True,'false':False,'1':True,'0':False})
assert stored_event.notna().all() and np.array_equal(true_event,stored_event.to_numpy(bool))
test['actual_exceeds_r']=true_event.astype(int)
assert np.isfinite(test.prob_mic_exceeds_r.to_numpy(float)).all()
assert test.prob_mic_exceeds_r.between(0,1).all()

for outer in range(1,6):
    train_ids=set(calib.loc[calib.outer_fold_for_calibration.eq(outer),'biosample'])
    test_ids=set(test.loc[test.outer_fold.eq(outer),'biosample'])
    assert not train_ids.intersection(test_ids),f'Outer-fold {outer} pathogen leakage'

scenarios=cutoffs.set_index('antibiotic')
for abx,frame in test.groupby('antibiotic'):
    expected=float(scenarios.loc[abx,'r_cutoff_mg_L_exclusive'])
    assert np.allclose(frame.r_cutoff_mg_L_exclusive.to_numpy(float),expected)
print(f'Checked {len(test):,} original NB54 held-out probabilities; 18 threshold scenarios.')
print(f'Checked {len(calib):,} training-only NB40 cross-fitted records.')
print('Outer-fold pathogen separation: PASSED.')


In [ ]:
#@title Cell 55.4 — Produce training-only probability/outcome pairs for recalibration
# Exact BMC-N grid and priors from NB41 / NB54, including analytic integration over b.
SIGMA_GRID=np.exp(np.linspace(np.log(.01),np.log(16.),4501))
LOG_SIGMA=np.log(SIGMA_GRID)
SIGMA2=SIGMA_GRID**2
PRIOR_RATE=.5

def fit_bmc_n_training_errors(errors):
    e=np.asarray(errors,dtype=np.float64)
    n=len(e)
    if n<15 or not np.isfinite(e).all():
        raise ValueError(f'BMC inner-training residual sample too small/non-finite (n={n})')
    mu=float(e.mean())
    sse=float(np.sum((e-mu)**2))
    log_density=(-(n-1)*LOG_SIGMA-.5*np.log(SIGMA2+n)-sse/(2*SIGMA2)
                 -n*mu**2/(2*(SIGMA2+n))-PRIOR_RATE*SIGMA_GRID+LOG_SIGMA)
    weights=np.exp(log_density-logsumexp(log_density))
    endpoint_mass=float(weights[:5].sum()+weights[-5:].sum())
    if endpoint_mass>1e-8:
        raise ValueError(f'Posterior grid truncated; endpoint mass={endpoint_mass}')
    means=n*mu/(n+SIGMA2)
    predictive_sd=np.sqrt(SIGMA2+SIGMA2/(n+SIGMA2))
    return weights,means,predictive_sd

def mixture_survival(thresholds_log2_error,posterior,chunk_size=96):
    weights,means,sds=posterior
    t=np.asarray(thresholds_log2_error,dtype=np.float64)
    result=np.empty(t.size,dtype=np.float64)
    for start in range(0,len(t),chunk_size):
        x=t[start:start+chunk_size,None]
        # ndtr(-z) is more stable than 1 - ndtr(z) for very small tail probabilities.
        tail=ndtr((means[None,:]-x)/sds[None,:])
        result[start:start+chunk_size]=tail@weights
    return np.clip(result,0,1)

calib_selected=calib.loc[calib.antibiotic.isin(scenarios.index)].copy()
prob_frames=[]
inner_audit=[]
for outer in range(1,6):
    outer_train=calib_selected.loc[calib_selected.outer_fold_for_calibration.eq(outer)]
    for abx,group in outer_train.groupby('antibiotic',sort=True):
        cutoff_log2=float(np.log2(scenarios.loc[abx,'r_cutoff_mg_L_exclusive']))
        for inner in (1,2,3):
            held_inner=group.loc[group.inner_fold.eq(inner)]
            fit_inner=group.loc[group.inner_fold.ne(inner)]
            if held_inner.empty:
                raise ValueError(f'No inner validation rows: {abx} outer={outer}, inner={inner}')
            # No target *residual* from held_inner is used to fit this BMC error distribution.
            assert not set(held_inner.interaction_row).intersection(set(fit_inner.interaction_row))
            assert not set(held_inner.biosample).intersection(set(fit_inner.biosample))
            posterior=fit_bmc_n_training_errors(fit_inner.training_residual_log2.to_numpy(float))
            resid_threshold=(cutoff_log2-held_inner.inner_oof_predicted_log2_mic.to_numpy(float))
            predicted=mixture_survival(resid_threshold,posterior)
            frame=held_inner[['outer_fold_for_calibration','inner_fold','interaction_row',
                              'biosample','antibiotic','observed_log2_mic']].copy()
            frame['raw_training_probability']=predicted
            frame['actual_exceeds_r']=(frame.observed_log2_mic.to_numpy(float)>cutoff_log2).astype(int)
            prob_frames.append(frame)
            inner_audit.append({'outer_fold':outer,'antibiotic':abx,'excluded_inner_fold':inner,
                                'n_fit_residuals':len(fit_inner),'n_calibration_targets':len(held_inner)})
    print(f'Completed outer fold {outer}/5: training-side cross-fitted probability pairs')

train_probs=pd.concat(prob_frames,ignore_index=True)
inner_audit=pd.DataFrame(inner_audit)
assert len(inner_audit)==5*18*3
assert len(train_probs)==len(calib_selected)
assert not train_probs.duplicated(['outer_fold_for_calibration','interaction_row']).any()
assert train_probs.raw_training_probability.between(0,1).all()
assert np.isfinite(train_probs.raw_training_probability.to_numpy(float)).all()
print(f'Training-only calibration pairs: {len(train_probs):,}; BMC inner models: {len(inner_audit)}.')
display(inner_audit.groupby('antibiotic')[['n_fit_residuals','n_calibration_targets']].min().head())


In [ ]:
#@title Cell 55.5 — Fit antibiotic-specific intercept-only probability recalibrators
# A fixed N(0, 2^2) regularisation penalty keeps sparse event groups stable.
# The slope of logit(p) is fixed to exactly 1, preserving within-antibiotic ranking.
LOGIT_EPS=1e-12
PRIOR_SD_ALPHA=2.0

def safe_logit(x):
    return logit(np.clip(np.asarray(x,dtype=float),LOGIT_EPS,1-LOGIT_EPS))

def intercept_recalibration_fit(probs,outcomes):
    x=safe_logit(probs)
    y=np.asarray(outcomes,dtype=int)
    assert x.size==y.size and x.size>=15
    def gradient(alpha):
        return float(np.sum(expit(x+alpha)-y)+alpha/PRIOR_SD_ALPHA**2)
    a=float(brentq(gradient,-35,35,xtol=1e-11))
    return a

calibration_params=[]
for (outer,abx),frame in train_probs.groupby(['outer_fold_for_calibration','antibiotic'],sort=True):
    y=frame.actual_exceeds_r.to_numpy(int)
    p=frame.raw_training_probability.to_numpy(float)
    alpha=intercept_recalibration_fit(p,y)
    n_pos=int(y.sum());n_neg=int(len(y)-n_pos)
    calibration_params.append({'outer_fold':int(outer),'antibiotic':abx,'n_training':len(y),
                               'n_training_exceeds_r':n_pos,'n_training_not_exceeds_r':n_neg,
                               'training_outcome_rate':float(y.mean()),
                               'training_mean_raw_probability':float(p.mean()),
                               'alpha_log_odds':alpha,
                               'sparse_event_warning':bool(min(n_pos,n_neg)<30),
                               'training_only_constant_prevalence':float((n_pos+.5)/(len(y)+1))})
params=pd.DataFrame(calibration_params)
assert len(params)==90 and not params.duplicated(['outer_fold','antibiotic']).any()
assert np.isfinite(params.alpha_log_odds.to_numpy(float)).all()
print('Calibrators fitted:',len(params),'(18 antibiotics × 5 outer folds).')
print('Sparse event calibrators (<30 positive or negative outcomes):',int(params.sparse_event_warning.sum()))
display(params.loc[params.antibiotic.isin(['cefepime','gentamicin']),
                   ['antibiotic','outer_fold','n_training','n_training_exceeds_r','alpha_log_odds','sparse_event_warning']].round(4))


In [ ]:
#@title Cell 55.6 — Independently evaluate raw vs recalibrated probabilities
compare=test.merge(params,how='left',on=['outer_fold','antibiotic'],validate='many_to_one')
if compare.alpha_log_odds.isna().any():
    raise ValueError('Missing fitted training-only recalibrator for some outer-test records')

compare['prob_original_bmc_n']=compare.prob_mic_exceeds_r.to_numpy(float)
compare['prob_recalibrated']=expit(safe_logit(compare.prob_original_bmc_n)+compare.alpha_log_odds.to_numpy(float))
compare['prob_training_prevalence_reference']=compare.training_only_constant_prevalence
assert compare.prob_recalibrated.between(0,1).all()
assert np.isfinite(compare.prob_recalibrated.to_numpy(float)).all()
assert len(compare)==len(test) and compare.interaction_row.is_unique
# The input BMC probability itself is never overwritten or updated.
assert np.array_equal(compare.prob_original_bmc_n.to_numpy(),compare.prob_mic_exceeds_r.to_numpy())

EVAL_EPS=1e-12

def metrics(y,p):
    y=np.asarray(y,dtype=int);p=np.asarray(p,dtype=float)
    q=np.clip(p,EVAL_EPS,1-EVAL_EPS)
    brier=float(np.mean((p-y)**2))
    loss=float(-np.mean(y*np.log(q)+(1-y)*np.log1p(-q)))
    bin_id=np.minimum(np.floor(np.clip(p,0,1)*10).astype(int),9)
    ece=sum(float((bin_id==i).mean()*abs(p[bin_id==i].mean()-y[bin_id==i].mean()))
            for i in range(10) if np.any(bin_id==i))
    return dict(brier=brier,log_loss=loss,ece_10=ece,
                mean_predicted=float(p.mean()),observed_rate=float(y.mean()),
                prediction_minus_observed=float(p.mean()-y.mean()))

def evaluate_group(frame,group_name):
    y=frame.actual_exceeds_r.to_numpy(int)
    p0=frame.prob_original_bmc_n.to_numpy(float)
    p1=frame.prob_recalibrated.to_numpy(float)
    pref=frame.prob_training_prevalence_reference.to_numpy(float)
    out={'group':group_name,'n_tested':len(y),'n_exceeds_r':int(y.sum()),
         'n_not_exceeds_r':int((1-y).sum())}
    m0=metrics(y,p0);m1=metrics(y,p1);mr=metrics(y,pref)
    for prefix,m in [('raw',m0),('recal',m1),('train_prevalence_reference',mr)]:
        out.update({f'{prefix}_{k}':v for k,v in m.items()})
    out['brier_change_recal_minus_raw']=m1['brier']-m0['brier']
    out['log_loss_change_recal_minus_raw']=m1['log_loss']-m0['log_loss']
    out['abs_bias_change_recal_minus_raw']=abs(m1['prediction_minus_observed'])-abs(m0['prediction_minus_observed'])
    for name,p in [('raw',p0),('recal',p1)]:
        high=p>=.9;low=p<=.1
        out[f'{name}_n_prob_ge_90']=int(high.sum())
        out[f'{name}_exceedance_fraction_in_ge_90']=float(y[high].mean()) if high.any() else np.nan
        out[f'{name}_n_prob_le_10']=int(low.sum())
        out[f'{name}_nonexceedance_fraction_in_le_10']=float((1-y[low]).mean()) if low.any() else np.nan
    return out

per_antibiotic=pd.DataFrame([evaluate_group(g,abx) for abx,g in compare.groupby('antibiotic',sort=True)])
pooled=pd.DataFrame([evaluate_group(compare,'all_18_descriptive')])
per_antibiotic['supported_n_positive_ge_30'] = per_antibiotic.n_exceeds_r.ge(30)

print('OVERALL — pooled across different EUCAST breakpoint scenarios (descriptive only):')
display(pooled[['n_tested','n_exceeds_r','raw_brier','recal_brier','brier_change_recal_minus_raw',
                'raw_log_loss','recal_log_loss','raw_prediction_minus_observed',
                'recal_prediction_minus_observed','raw_ece_10','recal_ece_10']].round(5))
print('ANTIBIOTIC-SPECIFIC comparison (negative changes indicate improvement):')
display(per_antibiotic[['group','n_tested','n_exceeds_r','raw_brier','recal_brier',
                         'brier_change_recal_minus_raw','raw_prediction_minus_observed',
                         'recal_prediction_minus_observed','raw_ece_10','recal_ece_10']].round(4))
print('Antibiotics with lower independent outer-test Brier score after recalibration:',
      int((per_antibiotic.brier_change_recal_minus_raw<0).sum()),'of 18.')


In [ ]:
#@title Cell 55.7 — Reliability bins and diagnostic before/after comparison
# Bin counts are reported so small or empty calibration bins are not overinterpreted.
def wilson_95(successes,n):
    if n==0:
        return np.nan,np.nan
    z=1.95996398454
    phat=successes/n
    d=1+z*z/n
    mid=(phat+z*z/(2*n))/d
    radius=z*np.sqrt(phat*(1-phat)/n+z*z/(4*n*n))/d
    return float(max(0,mid-radius)),float(min(1,mid+radius))

def bin_table(frame, group, probability_name, column):
    q=frame[column].to_numpy(float)
    y=frame.actual_exceeds_r.to_numpy(int)
    bin_id=np.minimum(np.floor(np.clip(q,0,1)*10).astype(int),9)
    rows=[]
    for i in range(10):
        m=bin_id==i;n=int(m.sum());n_pos=int(y[m].sum())
        lo,hi=wilson_95(n_pos,n)
        rows.append({'group':group,'method':probability_name,
                     'bin_low':i/10,'bin_high':(i+1)/10,'n':n,
                     'n_exceeds_r':n_pos,'mean_predicted':float(q[m].mean()) if n else np.nan,
                     'observed_fraction':float(y[m].mean()) if n else np.nan,
                     'wilson_lower95':lo,'wilson_upper95':hi})
    return pd.DataFrame(rows)

reliability=pd.concat([
    bin_table(frame,key,label,col)
    for key,frame in [('all_18_descriptive',compare)] + list(compare.groupby('antibiotic',sort=True))
    for label,col in [('raw','prob_original_bmc_n'),('recal','prob_recalibrated')]
],ignore_index=True)
assert len(reliability)==19*2*10
for (key,meth),group in reliability.groupby(['group','method']):
    expected=len(compare) if key=='all_18_descriptive' else int((compare.antibiotic==key).sum())
    assert int(group.n.sum())==expected

fig,axes=plt.subplots(1,3,figsize=(12,4),sharex=True,sharey=True)
for ax,(key,title) in zip(axes,[('all_18_descriptive','All 18 thresholds (descriptive)'),
                                ('cefepime','Cefepime'),('gentamicin','Gentamicin')]):
    ax.plot([0,1],[0,1],linestyle='--',linewidth=.8,color='.55')
    for meth,marker,color in [('raw','o','black'),('recal','x','.55')]:
        z=reliability.loc[reliability.group.eq(key)&reliability.method.eq(meth)&reliability.n.ge(10)]
        if len(z):
            ax.plot(z.mean_predicted,z.observed_fraction,linestyle='none',marker=marker,
                    color=color,markersize=4,label=('Original BMC-N' if meth=='raw' else 'Training-only recalibrated'))
    ax.set_title(title,fontsize=10)
    ax.set_xlim(0,1);ax.set_ylim(0,1)
    ax.set_xlabel('Predicted probability of exceeding threshold')
    ax.grid(color='.90',linewidth=.5)
axes[0].set_ylabel('Observed fraction exceeding threshold')
axes[-1].legend(frameon=False,fontsize=7,loc='lower right')
fig.tight_layout()
figfile=OUT/'55_BMCN_Probability_Recalibration_Reliability_Diagnostic.png'
fig.savefig(figfile,dpi=250,bbox_inches='tight')
plt.show()
print('Saved:',figfile)
print('Circles = original, crosses = recalibrated, dashed line = perfect calibration; display bins n ≥ 10.')


In [ ]:
#@title Cell 55.8 — Save results, source hashes and final validation audit
outputs={
  '55_BMCN_Training_Only_Inner_Probability_Pairs.csv.gz':train_probs,
  '55_BMCN_Inner_Posterior_Exclusion_Audit.csv':inner_audit,
  '55_BMCN_Recalibration_Parameters_By_Fold.csv':params,
  '55_BMCN_Probabilities_OuterTest_Original_Vs_Recalibrated.csv.gz':compare,
  '55_BMCN_Probability_Recalibration_Per_Antibiotic.csv':per_antibiotic,
  '55_BMCN_Probability_Recalibration_Overall_Descriptive.csv':pooled,
  '55_BMCN_Probability_Recalibration_Reliability_Bins.csv':reliability,
}
for filename,frame in outputs.items():
    destination=OUT/filename
    frame.to_csv(destination,index=False,compression=('gzip' if filename.endswith('.gz') else None))
    print('Saved:',filename,'rows:',len(frame))

provenance={
 'notebook':'NB55 v0.1',
 'purpose':'Train-only intercept-only log-odds recalibration of NB54 BMC-N MIC resistance-threshold exceedance probabilities',
 'sources':{name:str(path) for name,path in PATHS.items()},
 'source_sha256':{name:checksum(path) for name,path in PATHS.items()},
 'event':'Observed MIC > EUCAST R threshold selected by NB53 scenario; some thresholds apply only to restricted clinical indications',
 'validation':'5 independent outer test folds; no outer test outcomes used in training/fitting; raw NB54 probabilities unchanged',
 'training_procedure':'NB40 3-fold inner OOF predictions: for each inner fold, BMC-N posterior fitted on errors from the other two; use held-inner predictions and observed MIC for log-odds calibration',
 'training_qualification':'NB40 Model C inner fits share training pathogens; held-inner MICs can indirectly influence other inner residuals. Inner pairs are not a fully nested unbiased evaluation; independent outer-test metrics are decisive.',
 'recalibrator':'per antibiotic and outer fold intercept alpha; logit(p_recal) = logit(p_orig) + alpha, slope fixed 1; penalty alpha~N(0,2^2), fitted training-only',
 'bmc_posterior':'NB41/NB54 4501 log sigma grid 0.01–16, normal prior for b, exponential rate 0.5 for sigma; conditional b integrated analytically',
 'probability_scope':'event probabilities only; no recalibration of full continuous MIC posterior predictive distributions',
 'n_train_oof_pairs':len(train_probs),'n_independent_outer_test':len(compare),
 'n_antibiotics':len(per_antibiotic),'n_calibrators':len(params),
 'n_antibiotics_brier_improved':int((per_antibiotic.brier_change_recal_minus_raw<0).sum()),
 'clinical_caveats':'indication unknown; TMP-SMX coding and piperacillin-tazobactam ATU; confirm clinical utility and external validity before use',
}
path=OUT/'55_Provenance_And_Validation.json'
path.write_text(json.dumps(provenance,indent=2),encoding='utf-8')
assert figfile.is_file() and len(params)==90 and len(compare)==len(test)
print('Complete. Prior BMC notebooks, predictions, manuscript v0.6 and supplementary documents are unchanged.')
print('For review, upload:',OUT/'55_BMCN_Probability_Recalibration_Per_Antibiotic.csv')
print('And:',OUT/'55_BMCN_Probability_Recalibration_Overall_Descriptive.csv')
